# Multi-Agent SOC Architecture with Azure AI Foundry

**Goal:** Build a true Router-Worker incident triage system using Microsoft Agent Framework's GroupChatBuilder backed by Azure AI Foundry agents.

```
Alert In ──► SOC ROUTER (Alert Classifier)
                 │ (dynamically selects next agent)
                 ├── [if IOCs present] ──► Threat Intel Worker
                 ├── [if needs mapping] ──► Alert Enrichment Worker  
                 └── [always last] ──► SOC Reporter (synthesizes report)
```

**Pattern:** True Router-Worker — the SOC Router LLM dynamically decides which specialists to invoke per alert. Critical alerts get full analysis; Low-severity alerts get streamlined triage.

**SDK:** `agent-framework` + `agent-framework-azure-ai` (Microsoft Agent Framework, backed by Azure AI Foundry)

In [ ]:
!uv pip install agent-framework-azure-ai agent-framework-orchestrations nest-asyncio azure-identity python-dotenv rich matplotlib pandas -q

In [ ]:
import os
import json
from pathlib import Path
from typing import Annotated, Any

import nest_asyncio
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from dotenv import load_dotenv

from agent_framework import Agent, FunctionTool, tool
from agent_framework_orchestrations import GroupChatBuilder, GroupChatState, GroupChatSelectionFunction, TerminationCondition
from agent_framework.azure import AzureOpenAIResponsesClient
from azure.identity.aio import DefaultAzureCredential

nest_asyncio.apply()
load_dotenv(Path('..') / '.env', override=False)

AZURE_AI_PROJECT_ENDPOINT = os.getenv('AZURE_AI_PROJECT_ENDPOINT', '').strip()
MODEL_DEPLOYMENT_NAME = os.getenv('MODEL_DEPLOYMENT_NAME', 'gpt-4.1')
MOCK_MODE = not bool(AZURE_AI_PROJECT_ENDPOINT)

print(f"MODEL_DEPLOYMENT_NAME={MODEL_DEPLOYMENT_NAME}")
if MOCK_MODE:
    print('⚠️ MOCK_MODE=True (no AZURE_AI_PROJECT_ENDPOINT).')
    print('   🔵 cells run locally; 🔴 cells use simulated fallback output.')
else:
    print('✅ Azure endpoint detected. Foundry-backed agent cells can run.')

router_agent = None
ti_agent = None
enrichment_agent = None
reporter_agent = None
workflow = None
agent_registry = []
triage_runs = {}

ImportError: cannot import name 'TextResponseFormatConfigurationResponseFormatJsonObject' from 'azure.ai.projects.models' (c:\repo\agentic-soc-army\.venv\Lib\site-packages\azure\ai\projects\models\__init__.py)

## Visualize the Architecture

🔵 Works without Azure credentials.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))
fig.patch.set_facecolor('#0d1117')
ax.set_facecolor('#0d1117')
ax.set_xlim(0, 14)
ax.set_ylim(0, 10)
ax.axis('off')

def box(x, y, w, h, edge, title, subtitle=''):
    ax.add_patch(mpatches.FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.2', linewidth=2, edgecolor=edge, facecolor=edge + '22'))
    ax.text(x + w/2, y + h*0.62, title, ha='center', va='center', color=edge, fontsize=11, fontweight='bold')
    if subtitle:
        ax.text(x + w/2, y + h*0.28, subtitle, ha='center', va='center', color='#9aa4b2', fontsize=8)

box(4.6, 7.0, 4.8, 1.6, '#f0883e', 'SOC ROUTER', 'LLM orchestrator via GroupChatBuilder')
box(0.8, 4.3, 4.0, 1.5, '#4ecdc4', 'Threat Intel Worker', 'lookup_ioc + SIEM correlation')
box(5.0, 4.3, 4.0, 1.5, '#58a6ff', 'Alert Enrichment Worker', 'MITRE map + response playbook')
box(9.2, 4.3, 4.0, 1.5, '#7ee787', 'SOC Reporter', 'Final triage synthesis')

ax.annotate('', xy=(7.0, 8.9), xytext=(7.0, 9.8), arrowprops=dict(arrowstyle='->', color='#ff6b6b', lw=2.4))
ax.text(7.0, 9.9, 'Alert In', color='#ff6b6b', fontsize=11, ha='center', va='bottom', fontweight='bold')

ax.annotate('', xy=(2.8, 5.9), xytext=(6.6, 7.0), arrowprops=dict(arrowstyle='->', color='#4ecdc4', lw=2.0, connectionstyle='arc3,rad=0.12'))
ax.text(3.6, 6.9, '[if IOCs present]', color='#4ecdc4', fontsize=8)

ax.annotate('', xy=(7.0, 5.9), xytext=(7.0, 7.0), arrowprops=dict(arrowstyle='->', color='#58a6ff', lw=2.0))
ax.text(7.1, 6.4, '[if needs mapping]', color='#58a6ff', fontsize=8)

ax.annotate('', xy=(11.2, 5.9), xytext=(7.4, 7.0), arrowprops=dict(arrowstyle='->', color='#7ee787', lw=2.0, connectionstyle='arc3,rad=-0.12'))
ax.text(10.4, 6.9, '[always last]', color='#7ee787', fontsize=8)

ax.set_title('Router-Worker SOC Triage (Dynamic agent routing)', color='white', fontsize=13, pad=12)
plt.tight_layout()
plt.show()

## Load Sample Alerts

🔵 Works without Azure credentials.

In [ ]:
alerts_path = Path('data/sample_alerts.json')
if not alerts_path.exists():
    alerts_path = Path('../notebooks/data/sample_alerts.json')

with open(alerts_path, 'r', encoding='utf-8') as f:
    alerts = json.load(f)

df_alerts = pd.DataFrame(alerts)
severity_colors = {'Critical': '#ff4d4f', 'High': '#ff9f43', 'Medium': '#ffd166', 'Low': '#4dabf7'}

display(
    df_alerts[['alert_id', 'timestamp', 'severity', 'source', 'title']]
    .style
    .map(lambda v: f"color: {severity_colors.get(v, '#e6edf3')}", subset=['severity'])
    .set_properties(**{'background-color': '#161b22', 'color': '#e6edf3'})
)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
fig.patch.set_facecolor('#0d1117')
for ax in (ax1, ax2):
    ax.set_facecolor('#0d1117')

severity_order = ['Low', 'Medium', 'High', 'Critical']
sev_counts = df_alerts['severity'].value_counts().reindex(severity_order, fill_value=0)
ax1.bar(sev_counts.index, sev_counts.values, color=[severity_colors[s] for s in severity_order], edgecolor='#30363d')
ax1.set_title('Alerts by Severity', color='white')
ax1.tick_params(colors='white')
for spine in ax1.spines.values():
    spine.set_color('#30363d')

src_counts = df_alerts['source'].value_counts()
ax2.pie(src_counts.values, labels=src_counts.index, autopct='%1.0f%%', textprops={'color': 'white'}, colors=['#58a6ff', '#4ecdc4', '#7ee787'])
ax2.set_title('Alerts by Source', color='white')

plt.suptitle('SOC Alert Distribution Snapshot', color='white')
plt.tight_layout()
plt.show()

## Define Worker Tools

🔵 Works without Azure credentials.

Each worker gets explicit `FunctionTool` capabilities. We use the `@tool` decorator so functions can be bound as callable agent tools in the Agent Framework.

In [ ]:
@tool
def lookup_ioc_threat_intel(indicator: Annotated[str, 'IOC to look up — IP, domain, or hash']) -> str:
    """Look up an IOC in threat intelligence feeds."""
    mock_db = {
        '185.220.101.47': {'reputation': 'malicious', 'type': 'TOR exit node', 'confidence': 0.97, 'actor': 'Unknown/TOR', 'first_seen': '2022-03-01'},
        '194.165.16.101': {'reputation': 'malicious', 'type': 'C2 server', 'confidence': 0.95, 'actor': 'APT29', 'first_seen': '2024-11-12'}
    }
    result = mock_db.get(indicator, {'reputation': 'unknown', 'confidence': 0.0, 'type': 'no data'})
    return json.dumps({'indicator': indicator, **result})

@tool
def query_siem_related_alerts(alert_id: Annotated[str, 'Primary alert ID'], lookback_hours: Annotated[int, 'Hours to look back in SIEM'] = 48) -> str:
    """Query SIEM for alerts correlated with the given alert."""
    correlations = {
        'ALT-2025-001': [{'id': 'ALT-2025-005', 'reason': 'same user account', 'score': 0.82}],
        'ALT-2025-002': [{'id': 'ALT-2025-002', 'reason': 'same host', 'score': 0.91}]
    }
    return json.dumps({'alert_id': alert_id, 'lookback_hours': lookback_hours, 'correlated_alerts': correlations.get(alert_id, [])})

def _to_function_tool(fn: Any):
    try:
        return FunctionTool(fn)
    except Exception:
        try:
            return FunctionTool.from_callable(fn)
        except Exception:
            return fn

lookup_ioc_tool = _to_function_tool(lookup_ioc_threat_intel)
siem_query_tool = _to_function_tool(query_siem_related_alerts)

In [ ]:
@tool
def map_to_mitre_attack(alert_title: Annotated[str, 'Alert title'], description: Annotated[str, 'Alert description']) -> str:
    """Map an alert to MITRE ATT&CK tactics and techniques."""
    keywords = f"{alert_title} {description}".lower()
    mappings = []
    if 'credential' in keywords or 'stuffing' in keywords or 'login' in keywords:
        mappings.append({'tactic': 'Initial Access', 'technique': 'T1078', 'name': 'Valid Accounts'})
        mappings.append({'tactic': 'Credential Access', 'technique': 'T1110.004', 'name': 'Credential Stuffing'})
    if 'cobalt strike' in keywords or 'beacon' in keywords or 'c2' in keywords:
        mappings.append({'tactic': 'Command and Control', 'technique': 'T1071.001', 'name': 'Web Protocols'})
        mappings.append({'tactic': 'Defense Evasion', 'technique': 'T1055.012', 'name': 'Process Hollowing'})
    if 'powershell' in keywords or 'encoded' in keywords:
        mappings.append({'tactic': 'Execution', 'technique': 'T1059.001', 'name': 'PowerShell'})
        mappings.append({'tactic': 'Defense Evasion', 'technique': 'T1027', 'name': 'Obfuscated Files'})
    if not mappings:
        mappings.append({'tactic': 'Unknown', 'technique': 'TBD', 'name': 'Requires manual review'})
    return json.dumps({'alert_title': alert_title, 'mitre_mappings': mappings})

@tool
def get_response_playbook(attack_technique_id: Annotated[str, 'MITRE ATT&CK technique ID']) -> str:
    """Retrieve response playbook steps for a MITRE technique."""
    playbooks = {
        'T1078': {'priority': 'P1', 'steps': ['Revoke user sessions immediately', 'Force MFA re-registration', 'Block source IP', 'Export 48h sign-in logs', 'Check for persistence mechanisms']},
        'T1110.004': {'priority': 'P2', 'steps': ['Block originating IP range', 'Enable smart lockout if not active', 'Review auth logs for success-after-failure', 'Notify affected users']},
        'T1059.001': {'priority': 'P1', 'steps': ['Isolate affected endpoint', 'Capture memory dump', 'Decode base64 payload', 'Check run keys and scheduled tasks', 'Hunt for lateral movement']},
        'T1055.012': {'priority': 'P1', 'steps': ['Isolate host immediately', 'Capture network traffic', 'Identify parent process chain', 'Block C2 IP', 'Submit sample to sandbox']}
    }
    base = attack_technique_id.split('.')[0] if '.' in attack_technique_id else attack_technique_id
    result = playbooks.get(attack_technique_id, playbooks.get(base, {'priority': 'P3', 'steps': ['Investigate manually', 'Escalate to Tier 2']}))
    return json.dumps({'technique': attack_technique_id, **result})

mitre_map_tool = _to_function_tool(map_to_mitre_attack)
playbook_tool = _to_function_tool(get_response_playbook)

## Create Foundry-Backed Agents

We use `AzureAIProjectAgentProvider` to create agents backed by Azure AI Foundry. Each agent has tightly scoped instructions and tool access.

🔴 This cell requires Azure credentials unless `MOCK_MODE=True`.

In [ ]:
agent_registry = []

if MOCK_MODE:
    class MockAgent:
        def __init__(self, name: str, instructions: str, tools=None):
            self.name = name
            self.instructions = instructions
            self.tools = tools or []
            self.id = f"mock-{name}"

    router_agent = MockAgent('soc-router', 'Mock router')
    ti_agent = MockAgent('threat-intel-worker', 'Mock TI worker', [lookup_ioc_tool, siem_query_tool])
    enrichment_agent = MockAgent('alert-enrichment-worker', 'Mock enrichment worker', [mitre_map_tool, playbook_tool])
    reporter_agent = MockAgent('soc-reporter', 'Mock reporter')
    agent_registry = [
        {'role': 'router', 'name': router_agent.name, 'id': router_agent.id},
        {'role': 'threat_intel', 'name': ti_agent.name, 'id': ti_agent.id},
        {'role': 'enrichment', 'name': enrichment_agent.name, 'id': enrichment_agent.id},
        {'role': 'reporter', 'name': reporter_agent.name, 'id': reporter_agent.id}
    ]
    print('MOCK_MODE active: created mock agent descriptors.')
else:
    credential = DefaultAzureCredential()
    client = AzureOpenAIResponsesClient(
        project_endpoint=AZURE_AI_PROJECT_ENDPOINT,
        deployment_name=MODEL_DEPLOYMENT_NAME,
        credential=credential,
    )

    router_agent = Agent(
        name='soc-router',
        client=client,
        instructions=(
            "You are the SOC Alert Router. Analyze the incoming alert and decide which specialist agents should investigate. "
            "For each routing decision, name the agent that should speak next. Route to 'threat-intel-worker' when IOCs (IPs, domains, hashes) "
            "need investigation. Route to 'alert-enrichment-worker' when MITRE ATT&CK mapping and playbook retrieval is needed. "
            "After all specialists have reported, route to 'soc-reporter' to synthesize the final triage report. "
            "For Critical/High severity: route to BOTH specialists. For Medium: route based on alert content. "
            "For Low: route to enrichment only for quick classification, then reporter."
        ),
        tools=[],
    )

    ti_agent = Agent(
        name='threat-intel-worker',
        client=client,
        instructions=(
            "You are a SOC Threat Intelligence Analyst. Investigate IOCs using your tools and report findings as structured analysis. "
            "Always call lookup_ioc_threat_intel for each IOC and query_siem_related_alerts for correlation."
        ),
        tools=[lookup_ioc_tool, siem_query_tool],
    )

    enrichment_agent = Agent(
        name='alert-enrichment-worker',
        client=client,
        instructions=(
            "You are a SOC Alert Enrichment Specialist. Map alerts to MITRE ATT&CK using map_to_mitre_attack, "
            "then retrieve response playbooks using get_response_playbook for each identified technique."
        ),
        tools=[mitre_map_tool, playbook_tool],
    )

    reporter_agent = Agent(
        name='soc-reporter',
        client=client,
        instructions=(
            "You are the SOC Report Writer. Synthesize all preceding analyses into a structured Triage Report with: "
            "Executive Summary, IOC Analysis, MITRE Mapping, Severity Assessment, and Recommended Next Actions. "
            "Be precise and actionable - this goes to the on-call analyst."
        ),
        tools=[],
    )

    agent_registry = [
        {'role': 'router', 'name': router_agent.name, 'id': getattr(router_agent, 'id', 'responses-api')},
        {'role': 'threat_intel', 'name': ti_agent.name, 'id': getattr(ti_agent, 'id', 'responses-api')},
        {'role': 'enrichment', 'name': enrichment_agent.name, 'id': getattr(enrichment_agent, 'id', 'responses-api')},
        {'role': 'reporter', 'name': reporter_agent.name, 'id': getattr(reporter_agent, 'id', 'responses-api')},
    ]

    print('Created agents via AzureOpenAIResponsesClient (Responses API).')

pd.DataFrame(agent_registry)

## Build the Router-Worker GroupChat

🔴 Uses real agents when Azure is configured; otherwise runs with mock fallback.

In [ ]:
if not MOCK_MODE:
    workflow = GroupChatBuilder(
        participants=[ti_agent, enrichment_agent, reporter_agent],
        orchestrator_agent=router_agent,
        max_rounds=10,
        intermediate_outputs=True,
    ).build()
    print('Router-Worker workflow built (max_rounds=10, intermediate_outputs=True).')
else:
    workflow = None
    print('MOCK_MODE: workflow build skipped (mock agents are not real Agent instances).')

## Triage Alert 1: Cobalt Strike Beacon (Critical)

🔴 Uses real GroupChat when Azure is configured; otherwise runs simulated routing.

In [ ]:
async def run_triage(alert_id: str) -> dict:
    alert = next(a for a in alerts if a['alert_id'] == alert_id)
    input_message = (
        f"Alert ID: {alert['alert_id']}\n"
        f"Severity: {alert['severity']}\n"
        f"Source: {alert['source']}\n"
        f"Title: {alert['title']}\n"
        f"Description: {alert['description']}\n"
        f"Indicators: {', '.join(alert.get('indicators', []))}\n"
        f"Affected Users: {', '.join(alert.get('affected_users', []))}\n"
        f"Raw log: {alert.get('raw_log', '')}\n"
    )

    if MOCK_MODE:
        if alert['severity'] in ('Critical', 'High'):
            route = ['soc-router', 'threat-intel-worker', 'alert-enrichment-worker', 'soc-reporter']
        else:
            route = ['soc-router', 'alert-enrichment-worker', 'soc-reporter']

        report = f"""TRIAGE REPORT ({alert['alert_id']})\n\n
Executive Summary: Simulated Router-Worker analysis for {alert['title']}.\n
IOC Analysis: {'Full IOC + SIEM correlation executed.' if 'threat-intel-worker' in route else 'IOC deep-dive skipped for streamlined low-severity triage.'}\n
MITRE Mapping: Generated ATT&CK mapping and response playbook alignment.\n
Severity Assessment: {alert['severity']}\n
Recommended Next Actions: Follow SOC response playbook and monitor related entities."""

        return {
            'alert_id': alert_id,
            'severity': alert['severity'],
            'route': route,
            'events': [{'agent_name': r, 'content': f'simulated event from {r}'} for r in route],
            'report': report
        }

    # --- Live Foundry execution ---
    events_raw = []
    route = []
    report = ''

    stream = workflow.run(input_message, stream=True)
    async for event in stream:
        events_raw.append(event)
        # Track agent routing via executor_invoked events
        eid = getattr(event, 'executor_id', None) or ''
        etype = getattr(event, 'type', '')
        if etype == 'executor_invoked' and eid and (not route or route[-1] != eid):
            route.append(eid)
        # Capture output data as report text
        if etype == 'output':
            data = getattr(event, 'data', None)
            if isinstance(data, str) and data.strip():
                report = data

    # Also try the final result
    try:
        result = await stream.get_final_response()
        outputs = result.get_outputs() if hasattr(result, 'get_outputs') else []
        for o in outputs:
            if isinstance(o, str) and o.strip():
                report = o
    except Exception:
        pass

    if not report:
        report = 'No final report text extracted. Inspect events_raw for details.'

    return {
        'alert_id': alert_id,
        'severity': alert['severity'],
        'route': route,
        'events': [{'type': getattr(e, 'type', '?'), 'executor_id': getattr(e, 'executor_id', '')} for e in events_raw],
        'report': report
    }

run_critical = await run_triage('ALT-2025-002')
triage_runs['ALT-2025-002'] = run_critical

print('Routing path:', ' → '.join(run_critical['route']))
print()
print(run_critical['report'])

## Triage Alert 2: Impossible Travel (Low)

🔴 Uses real GroupChat when Azure is configured; otherwise runs simulated routing.

In [ ]:
run_low = await run_triage('ALT-2025-005')
triage_runs['ALT-2025-005'] = run_low

print('Routing path:', ' → '.join(run_low['route']))
print()
print(run_low['report'])

## Visualize Routing Decisions

🔵 Works without Azure credentials.

In [ ]:
crit = triage_runs['ALT-2025-002']
low = triage_runs['ALT-2025-005']

all_agents = ['soc-router', 'threat-intel-worker', 'alert-enrichment-worker', 'soc-reporter']
crit_flags = [1 if a in crit['route'] else 0 for a in all_agents]
low_flags = [1 if a in low['route'] else 0 for a in all_agents]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.patch.set_facecolor('#0d1117')
for ax in axes:
    ax.set_facecolor('#0d1117')

axes[0].barh(all_agents, crit_flags, color=['#f0883e', '#4ecdc4', '#58a6ff', '#7ee787'])
axes[0].set_title('Critical Alert Route (ALT-2025-002)', color='white')
axes[0].set_xlim(0, 1.2)
axes[0].tick_params(colors='white')

axes[1].barh(all_agents, low_flags, color=['#f0883e', '#4ecdc4', '#58a6ff', '#7ee787'])
axes[1].set_title('Low Alert Route (ALT-2025-005)', color='white')
axes[1].set_xlim(0, 1.2)
axes[1].tick_params(colors='white')

for ax in axes:
    for spine in ax.spines.values():
        spine.set_color('#30363d')

plt.suptitle('Dynamic Router decisions by alert severity/context', color='white')
plt.tight_layout()
plt.show()

print('Critical route:', ' → '.join(crit['route']))
print('Low route     :', ' → '.join(low['route']))
print('Report sizes  :', len(crit['report']), '(critical),', len(low['report']), '(low)')

## Agent Call Chain Comparison

🔵 Works without Azure credentials.

In [ ]:
comparison = pd.DataFrame([
    {
        'alert_id': 'ALT-2025-002',
        'severity': triage_runs['ALT-2025-002']['severity'],
        'call_chain': ' → '.join(triage_runs['ALT-2025-002']['route']),
        'agent_count': len(triage_runs['ALT-2025-002']['route']),
        'report_chars': len(triage_runs['ALT-2025-002']['report'])
    },
    {
        'alert_id': 'ALT-2025-005',
        'severity': triage_runs['ALT-2025-005']['severity'],
        'call_chain': ' → '.join(triage_runs['ALT-2025-005']['route']),
        'agent_count': len(triage_runs['ALT-2025-005']['route']),
        'report_chars': len(triage_runs['ALT-2025-005']['report'])
    }
])

display(comparison)

print('Expected pattern check:')
print('- Critical alert: Router → TI → Enrichment → Reporter')
print('- Low alert     : Router → Enrichment → Reporter')

## Cleanup

🔴 Deletes remote agents when Azure is configured; otherwise no-op in mock mode.

In [ ]:
if MOCK_MODE:
    print('MOCK_MODE cleanup: no remote agents to delete.')
else:
    # AzureOpenAIResponsesClient agents are stateless - no server-side cleanup needed.
    # Close the credential if it was stored.
    print('Responses API agents are stateless. No server-side cleanup required.')
    print(f'Agents used: {[r["name"] for r in agent_registry]}')